In [30]:
import json
import pandas as pd
import os
import re
from typing import List, Dict, Union
from datetime import datetime

In [50]:
test_data = '''{
    "职位名称": "硬件开发3（工作地武汉）",
    "公司名称": "中国船舶集团有限公司第七一〇研究所",
    "经纬度": [
        "30.390866",
        "114.348331"
    ],
    "jobid": "168090596",
    "薪资": "1.5-1.6万/年",
    "城市": "武汉-江夏区",
    "学历要求": "硕士",
    "经验要求": "在校生/应届生",
    "公司性质": "事业单位",
    "行业": "学术/科研",
    "发布时间": "2025-12-08 17:56:18"
}
''' 

In [2]:
def read_json_data(file_path: str) -> pd.DataFrame:
    """
    读取JSON文件并转换为DataFrame

    参数:
        file_path: JSON文件路径
    
    返回:
        pd.DataFrame: 包含职位数据的DataFrame
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    df = pd.DataFrame(data)
    return df


def read_latest_json(data_dir: str = '../data') -> pd.DataFrame:
    """
    读取data目录下最新的JSON文件
    
    参数:
        data_dir: 数据目录路径，默认为'../data'
    
    返回:
        pd.DataFrame: 包含职位数据的DataFrame
    """
    # 获取所有JSON文件
    json_files = [f for f in os.listdir(data_dir) if f.endswith('.json')]
    
    if not json_files:
        raise FileNotFoundError(f"在 {data_dir} 目录下没有找到JSON文件")
    
    # 按修改时间排序，获取最新的文件
    latest_file = max(json_files, key=lambda f: os.path.getmtime(os.path.join(data_dir, f)))
    
    file_path = os.path.join(data_dir, latest_file)
    print(f"读取文件: {latest_file}")
    
    return read_json_data(file_path)


def read_all_json_files(data_dir: str = 'data') -> pd.DataFrame:
    """
    读取data目录下所有JSON文件并合并
    
    参数:
        data_dir: 数据目录路径，默认为'data'
    
    返回:
        pd.DataFrame: 合并后的DataFrame
    """
    json_files = [f for f in os.listdir(data_dir) if f.endswith('.json')]
    
    if not json_files:
        raise FileNotFoundError(f"在 {data_dir} 目录下没有找到JSON文件")
    
    all_data = []
    for file in json_files:
        file_path = os.path.join(data_dir, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            all_data.extend(data)
    
    df = pd.DataFrame(all_data)
    print(f"共读取 {len(json_files)} 个文件，合计 {len(df)} 条数据")
    
    return df


In [57]:
def clean_salary(salary_str):
    """
    清洗薪资数据，统一转换为k/月
    
    支持的格式：
    - 4-6千
    - 1.2-2万
    - 8千-1.6万
    - 10-15万/年
    - 8千-1.6万·14薪
    - 15k-20k
    - 50/天
    - 200-300/天
    - 面议
    """
    if pd.isna(salary_str) or not salary_str:
        return {'min_salary': None, 'max_salary': None, 'avg_salary': None}
    
    salary_str = str(salary_str).strip()
    
    # 处理特殊情况：面议、待遇优厚等
    if any(keyword in salary_str for keyword in ['面议', '待遇', '薪资面议', '面谈', '协商']):
        return {'min_salary': None, 'max_salary': None, 'avg_salary': None}
    
    # 单位转换辅助函数
    def convert_to_k_per_month(val, unit, is_yearly=False, is_daily=False, num_months=12):
        """
        将薪资统一转换为k/月
        
        参数:
            val: 数值
            unit: 单位（千/万/k/K）
            is_yearly: 是否为年薪
            is_daily: 是否为日薪
            num_months: 月数（如14薪就是14）
        """
        val = float(val)
        
        # 如果是日薪，先转为月薪（按22个工作日计算）
        if is_daily:
            # 日薪默认单位是元，需要转为千
            val_k = (val * 22) / 1000
        else:
            # 先转换单位到k
            if unit in ['万', 'W', 'w']:
                val_k = val * 10
            else:  # 千, k, K
                val_k = val
            
            # 如果是年薪，除以12转为月薪
            if is_yearly:
                val_k = val_k / 12
            # 如果有多薪制，除以月数
            elif num_months > 12:
                val_k = val_k * 12 / num_months
        
        return val_k
    
    # ===== 优先检测日薪格式 =====
    # 模式: 日薪 - "50/天" "200-300/天" "150元/天"
    is_daily = bool(re.search(r'/天|每天|日薪|/日', salary_str))
    
    if is_daily:
        # 移除干扰字符
        salary_core = re.sub(r'/天|每天|日薪|/日|元', '', salary_str).strip()
        
        # 日薪区间: "200-300"
        pattern_daily_range = r'([\d.]+)\s*[-~至]\s*([\d.]+)'
        match_daily_range = re.search(pattern_daily_range, salary_core)
        
        if match_daily_range:
            min_val = match_daily_range.group(1)
            max_val = match_daily_range.group(2)
            
            min_salary = convert_to_k_per_month(min_val, None, is_daily=True)
            max_salary = convert_to_k_per_month(max_val, None, is_daily=True)
            avg_salary = (min_salary + max_salary) / 2
            
            return {
                'min_salary': round(min_salary, 1),
                'max_salary': round(max_salary, 1),
                'avg_salary': round(avg_salary, 1)
            }
        
        # 单一日薪: "150"
        pattern_daily_single = r'([\d.]+)'
        match_daily_single = re.search(pattern_daily_single, salary_core)
        
        if match_daily_single:
            val = match_daily_single.group(1)
            salary = convert_to_k_per_month(val, None, is_daily=True)
            
            return {
                'min_salary': round(salary, 1),
                'max_salary': round(salary, 1),
                'avg_salary': round(salary, 1)
            }
    
    # 检测是否为年薪
    is_yearly = bool(re.search(r'/年|每年|年薪', salary_str))
    
    # 提取薪资月数（如14薪、13薪）
    months_match = re.search(r'[·\*×xX]?\s*(\d+)\s*薪', salary_str)
    num_months = int(months_match.group(1)) if months_match else 12
    
    # 移除干扰字符，保留核心薪资部分
    salary_core = re.sub(r'[·\*×xX]\s*\d+\s*薪', '', salary_str)
    salary_core = re.sub(r'/年|每年|年薪|/月|每月|月薪|元', '', salary_core)
    salary_core = salary_core.strip()
    
    # ===== 模式匹配（按优先级） =====
    
    # 模式1: 跨单位区间 - "8千-1.6万" "1.2k-2万"
    pattern1 = r'([\d.]+)\s*([千万kKwW])\s*[-~至]\s*([\d.]+)\s*([千万kKwW])'
    match1 = re.search(pattern1, salary_core)
    
    if match1:
        min_val = match1.group(1)
        min_unit = match1.group(2)
        max_val = match1.group(3)
        max_unit = match1.group(4)
        
        min_salary = convert_to_k_per_month(min_val, min_unit, is_yearly, False, num_months)
        max_salary = convert_to_k_per_month(max_val, max_unit, is_yearly, False, num_months)
        avg_salary = (min_salary + max_salary) / 2
        
        return {
            'min_salary': round(min_salary, 1),
            'max_salary': round(max_salary, 1),
            'avg_salary': round(avg_salary, 1)
        }
    
    # 模式2: 同单位区间 - "4-6千" "1.2-2万" "10-15k"
    pattern2 = r'([\d.]+)\s*[-~至]\s*([\d.]+)\s*([千万kKwW])'
    match2 = re.search(pattern2, salary_core)
    
    if match2:
        min_val = match2.group(1)
        max_val = match2.group(2)
        unit = match2.group(3)
        
        min_salary = convert_to_k_per_month(min_val, unit, is_yearly, False, num_months)
        max_salary = convert_to_k_per_month(max_val, unit, is_yearly, False, num_months)
        avg_salary = (min_salary + max_salary) / 2
        
        return {
            'min_salary': round(min_salary, 1),
            'max_salary': round(max_salary, 1),
            'avg_salary': round(avg_salary, 1)
        }
    
    # 模式3: 纯数字区间（无单位，默认千） - "4-6"
    pattern3 = r'^([\d.]+)\s*[-~至]\s*([\d.]+)$'
    match3 = re.search(pattern3, salary_core)
    
    if match3:
        min_val = match3.group(1)
        max_val = match3.group(2)
        
        min_salary = convert_to_k_per_month(min_val, '千', is_yearly, False, num_months)
        max_salary = convert_to_k_per_month(max_val, '千', is_yearly, False, num_months)
        avg_salary = (min_salary + max_salary) / 2
        
        return {
            'min_salary': round(min_salary, 1),
            'max_salary': round(max_salary, 1),
            'avg_salary': round(avg_salary, 1)
        }
    
    # 模式4: 单一数值 - "8千" "2万" "15k"
    pattern4 = r'([\d.]+)\s*([千万kKwW])'
    match4 = re.search(pattern4, salary_core)
    
    if match4:
        val = match4.group(1)
        unit = match4.group(2)
        
        salary = convert_to_k_per_month(val, unit, is_yearly, False, num_months)
        
        return {
            'min_salary': round(salary, 1),
            'max_salary': round(salary, 1),
            'avg_salary': round(salary, 1)
        }
    
    # 无法解析
    return {'min_salary': None, 'max_salary': None, 'avg_salary': None}

In [55]:
def clean_job_data(df, days_limit=None, remove_duplicates=True):
    """
    完整的职位数据清洗函数
    
    参数:
        df: 原始DataFrame
        days_limit: 保留最近N天的数据，None表示不限制
        remove_duplicates: 是否去除重复数据
    
    返回:
        pd.DataFrame: 清洗后的数据
    """
    # 创建副本，避免修改原始数据
    df_clean = df.copy()
    
    # ===== 重要：重置索引，避免索引对齐问题 =====
    df_clean = df_clean.reset_index(drop=True)
    
    print(f"原始数据: {len(df_clean)} 条")
    
    # 记录原始列
    original_columns = df_clean.columns.tolist()
    original_column_count = len(original_columns)
    
    # 1. 去除重复数据（基于jobid）
    if remove_duplicates and 'jobid' in df_clean.columns:
        before_count = len(df_clean)
        df_clean = df_clean.drop_duplicates(subset=['jobid'], keep='first')
        # ===== 重要：去重后重置索引 =====
        df_clean = df_clean.reset_index(drop=True)
        removed = before_count - len(df_clean)
        print(f"去除重复数据: {removed} 条，剩余 {len(df_clean)} 条")
    
    # 2. 清洗薪资数据
    if '薪资' in df_clean.columns:
        print("正在清洗薪资数据...")
        
        # ===== 修复：逐行处理，确保索引对齐 =====
        salary_results = []
        failed_cases = []
        
        for idx, salary_value in enumerate(df_clean['薪资']):
            try:
                result = clean_salary(salary_value)
                salary_results.append(result)
            except Exception as e:
                # 记录失败的案例
                failed_cases.append({
                    'index': idx,
                    'salary': salary_value,
                    'error': str(e)
                })
                salary_results.append({'min_salary': None, 'max_salary': None, 'avg_salary': None})
        
        # 转换为DataFrame
        salary_df = pd.DataFrame(salary_results)
        
        # ===== 验证数据长度一致 =====
        if len(salary_df) != len(df_clean):
            print(f"警告：薪资数据长度({len(salary_df)})与原数据({len(df_clean)})不一致！")
            return df_clean
        
        # 直接赋值（因为索引已经对齐）
        df_clean['薪资下限(k)'] = salary_df['min_salary'].values
        df_clean['薪资上限(k)'] = salary_df['max_salary'].values
        df_clean['平均薪资(k)'] = salary_df['avg_salary'].values
        
        # 统计薪资清洗结果
        valid_salary = df_clean['平均薪资(k)'].notna().sum()
        invalid_salary = df_clean['平均薪资(k)'].isna().sum()
        print(f"薪资数据清洗完成: {valid_salary}/{len(df_clean)} 条有效, {invalid_salary} 条无效")
        
        # 如果有失败案例，打印出来
        if failed_cases:
            print(f"\n警告：有 {len(failed_cases)} 条薪资解析失败:")
            for case in failed_cases[:5]:  # 只显示前5条
                print(f"  索引 {case['index']}: '{case['salary']}' - 错误: {case['error']}")
            if len(failed_cases) > 5:
                print(f"  ... 还有 {len(failed_cases) - 5} 条")
        
        # ===== 调试：显示一些无效薪资的样本 =====
        if invalid_salary > 0:
            print("\n无效薪资样本（前10条）:")
            invalid_samples = df_clean[df_clean['平均薪资(k)'].isna()]['薪资'].head(10)
            for idx, salary in enumerate(invalid_samples, 1):
                print(f"  {idx}. '{salary}'")
    
    # 3. 处理发布时间
    if '发布时间' in df_clean.columns:
        df_clean['发布时间'] = pd.to_datetime(df_clean['发布时间'], errors='coerce')
        
        if days_limit is not None:
            from datetime import datetime, timedelta
            cutoff_date = datetime.now() - timedelta(days=days_limit)
            before_count = len(df_clean)
            df_clean = df_clean[df_clean['发布时间'] >= cutoff_date]
            # ===== 重要：筛选后重置索引 =====
            df_clean = df_clean.reset_index(drop=True)
            removed = before_count - len(df_clean)
            print(f"筛选最近 {days_limit} 天数据: 移除 {removed} 条，剩余 {len(df_clean)} 条")
    
    # 4. 处理经纬度数据
    if '经纬度' in df_clean.columns:
        df_clean['纬度'] = df_clean['经纬度'].apply(
            lambda x: float(x[0]) if isinstance(x, list) and len(x) > 0 and x[0] else None
        )
        df_clean['经度'] = df_clean['经纬度'].apply(
            lambda x: float(x[1]) if isinstance(x, list) and len(x) > 1 and x[1] else None
        )
    
    # 5. 清理城市字段
    if '城市' in df_clean.columns:
        df_clean['主城市'] = df_clean['城市'].str.split('-').str[0]
        df_clean['区县'] = df_clean['城市'].str.split('-').str[1]
    
    # 6. 处理学历要求
    if '学历要求' in df_clean.columns:
        education_map = {
            '不限': 0,
            '初中': 1,
            '中专': 2,
            '高中': 2,
            '大专': 3,
            '本科': 4,
            '硕士': 5,
            '博士': 6
        }
        df_clean['学历等级'] = df_clean['学历要求'].map(education_map)
    
    # 7. 处理经验要求
    if '经验要求' in df_clean.columns:
        df_clean['经验年限'] = df_clean['经验要求'].str.extract(r'(\d+)').astype(float)
        df_clean['经验年限'] = df_clean['经验年限'].fillna(0)
    
    # 8. 移除空值过多的行（只考虑原始列）
    null_counts = df_clean[original_columns].isnull().sum(axis=1)
    max_nulls = original_column_count * 0.5
    before_count = len(df_clean)
    df_clean = df_clean[null_counts <= max_nulls]
    # ===== 重要：筛选后重置索引 =====
    df_clean = df_clean.reset_index(drop=True)
    removed = before_count - len(df_clean)
    if removed > 0:
        print(f"移除空值过多的行: {removed} 条")
    
    print(f"\n最终数据: {len(df_clean)} 条")
    if '平均薪资(k)' in df_clean.columns:
        valid_pct = df_clean['平均薪资(k)'].notna().sum() / len(df_clean) * 100
        print(f"薪资数据完整性: {valid_pct:.1f}%")
    
    return df_clean

In [31]:
def save_cleaned_data(df, output_dir='../washdata', file_prefix='cleaned_jobs'):
    """
    保存清洗后的数据到指定目录
    
    参数:
        df: 清洗后的DataFrame
        output_dir: 输出目录路径
        file_prefix: 文件名前缀
    
    返回:
        dict: 保存的文件路径字典
    """
    # 获取绝对路径
    output_dir = os.path.abspath(output_dir)
    
    # 创建目录（如果不存在）
    os.makedirs(output_dir, exist_ok=True)
    print(f"数据将保存到: {output_dir}")
    
    # 添加时间戳
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # 定义文件路径
    files = {
        'json': os.path.join(output_dir, f'{file_prefix}_{timestamp}.json'),
        'csv': os.path.join(output_dir, f'{file_prefix}_{timestamp}.csv'),
        'excel': os.path.join(output_dir, f'{file_prefix}_{timestamp}.xlsx'),
    }
    
    # 保存为JSON
    df.to_json(files['json'], orient='records', force_ascii=False, indent=4)
    print(f"✓ JSON文件已保存: {os.path.basename(files['json'])}")
    
    # 保存为CSV（UTF-8 with BOM，Excel可以正确打开中文）
    df.to_csv(files['csv'], index=False, encoding='utf-8-sig')
    print(f"✓ CSV文件已保存: {os.path.basename(files['csv'])}")
    
    # 保存为Excel
    try:
        df.to_excel(files['excel'], index=False, engine='openpyxl')
        print(f"✓ Excel文件已保存: {os.path.basename(files['excel'])}")
    except Exception as e:
        print(f"✗ Excel保存失败: {e}")
        files['excel'] = None
    
    # 同时保存一个最新版本（不带时间戳，方便后续直接使用）
    latest_files = {
        'json': os.path.join(output_dir, f'{file_prefix}_latest.json'),
        'csv': os.path.join(output_dir, f'{file_prefix}_latest.csv'),
        'excel': os.path.join(output_dir, f'{file_prefix}_latest.xlsx'),
    }
    
    df.to_json(latest_files['json'], orient='records', force_ascii=False, indent=4)
    df.to_csv(latest_files['csv'], index=False, encoding='utf-8-sig')
    try:
        df.to_excel(latest_files['excel'], index=False, engine='openpyxl')
    except:
        pass
    
    print(f"\n✓ 最新版本已保存为 '{file_prefix}_latest.*'")
    
    # 保存数据统计信息
    stats_file = os.path.join(output_dir, f'data_stats_{timestamp}.txt')
    with open(stats_file, 'w', encoding='utf-8') as f:
        f.write(f"数据清洗统计报告\n")
        f.write(f"=" * 50 + "\n")
        f.write(f"清洗时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"总记录数: {len(df)}\n\n")
        
        f.write(f"数据概览:\n")
        f.write(f"-" * 50 + "\n")
        f.write(df.describe().to_string())
        f.write(f"\n\n")
        
        f.write(f"字段信息:\n")
        f.write(f"-" * 50 + "\n")
        info_buffer = df.info(buf=None, verbose=True)
        f.write(str(info_buffer))
        
        if '平均薪资(k)' in df.columns:
            f.write(f"\n\n薪资统计:\n")
            f.write(f"-" * 50 + "\n")
            f.write(f"平均薪资: {df['平均薪资(k)'].mean():.2f}k\n")
            f.write(f"中位数薪资: {df['平均薪资(k)'].median():.2f}k\n")
            f.write(f"最高薪资: {df['薪资上限(k)'].max():.2f}k\n")
            f.write(f"最低薪资: {df['薪资下限(k)'].min():.2f}k\n")
    
    print(f"✓ 统计报告已保存: {os.path.basename(stats_file)}")
    
    print(f"\n" + "=" * 50)
    print(f"所有文件保存完成！共 {len(df)} 条记录")
    print(f"保存路径: {output_dir}")
    print("=" * 50)
    
    return files

In [56]:
path="../data/jobs_data_20251228_190155.json"
df=read_json_data(path)
clean_job_data(df)

原始数据: 60 条
去除重复数据: 4 条，剩余 56 条
正在清洗薪资数据...
薪资数据清洗完成: 54/56 条有效, 2 条无效

无效薪资样本（前10条）:
  1. ''
  2. ''

最终数据: 56 条
薪资数据完整性: 96.4%


,职位名称,公司名称,经纬度,jobid,薪资,城市,学历要求,经验要求,公司性质,行业,发布时间,薪资下限(k),薪资上限(k),平均薪资(k),纬度,经度,主城市,区县,学历等级,经验年限
0,嵌入式软件工程师-成都(J10266),中科芯集成电路,"[0.000000, 0.000000]",167102488,1.2-2万,成都,硕士,在校生/应届生,国企,电子技术/半导体/集成电路,2025-10-20 10:28:30,12.0,20.0,16.0,0.000000,0.000000,成都,NaN,5,0.0
1,嵌入式软件开发（浙江绍兴）（成都）,浙江捷昌线性驱动科技,"[29.525223, 120.842252]",168433045,8千-1.5万,绍兴-新昌县,本科,在校生/应届生,民营,机械/设备/重工,2025-09-30 11:08:02,8.0,15.0,11.5,29.525223,120.842252,绍兴,新昌县,4,0.0
2,嵌入式硬件工程师-成都(J10265),中科芯集成电路,"[0.000000, 0.000000]",167102579,1.2-2万,成都,硕士,在校生/应届生,国企,电子技术/半导体/集成电路,2025-10-20 10:28:29,12.0,20.0,16.0,0.000000,0.000000,成都,NaN,5,0.0
3,嵌入式助理工程师（软件&硬件）,昆山丘钛微电子科技,"[31.34106, 120.890399]",168561176,6千-1万,昆山,本科,在校生/应届生,已上市,电子技术/半导体/集成电路,2025-11-16 23:36:00,6.0,10.0,8.0,31.341060,120.890399,昆山,NaN,4,0.0
4,软件开发（成都）,苏州旭创科技,"[30.766059, 103.906372]",167370431,1.8-3.6万·15薪,成都-高新区,硕士,在校生/应届生,已上市,电子技术/半导体/集成电路,2025-10-09 16:30:45,14.4,28.8,21.6,30.766059,103.906372,成都,高新区,5,0.0
5,嵌入式软件开发4（工作地武汉）,中国船舶集团有限公司第七一〇研究所,"[30.390866, 114.348331]",168094926,1.5-1.6万,武汉-江夏区,硕士,在校生/应届生,事业单位,学术/科研,2025-12-08 17:55:41,15.0,16.0,15.5,30.390866,114.348331,武汉,江夏区,5,0.0
6,Linux系统软件工程师（有经验）,成都熊谷加世电器,"[30.764109, 103.916966]",159507828,18-28万/年,成都-郫都区,硕士,1年及以上,民营,机械/设备/重工,2025-11-22 13:09:16,15.0,23.3,19.2,30.764109,103.916966,成都,郫都区,5,1.0
7,嵌入式软件工程师（地点：上海）（成都）,苏州顺芯半导体,"[31.186858, 121.43842]",161144177,1.4-2.2万,上海-徐汇区,本科,在校生/应届生,民营,电子技术/半导体/集成电路,2025-12-26 08:51:51,14.0,22.0,18.0,31.186858,121.438420,上海,徐汇区,4,0.0
8,嵌入式软件设计师（成都）,中国航空无线电电子研究所,"[30.690852, 103.963028]",168008933,1-2万,成都-青羊区,硕士,在校生/应届生,事业单位,政府/公共事业,2025-12-27 04:23:32,10.0,20.0,15.0,30.690852,103.963028,成都,青羊区,5,0.0
9,算法助理工程师（嵌入式方向）,昆山丘钛微电子科技,"[31.34106, 120.890399]",168561189,6-8千,昆山,本科,在校生/应届生,已上市,电子技术/半导体/集成电路,2025-11-16 23:35:58,6.0,8.0,7.0,31.341060,120.890399,昆山,NaN,4,0.0


In [34]:
save_cleaned_data(wash_df)

数据将保存到: C:\Users\Buer_vakabauta\Desktop\Course\data_analysis\src\washdata
✓ JSON文件已保存: cleaned_jobs_20251228_211500.json
✓ CSV文件已保存: cleaned_jobs_20251228_211500.csv
✗ Excel保存失败: No module named 'openpyxl'

✓ 最新版本已保存为 'cleaned_jobs_latest.*'
<class 'pandas.core.frame.DataFrame'>
Index: 56 entries, 0 to 59
Data columns (total 20 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   职位名称     56 non-null     object        
 1   公司名称     56 non-null     object        
 2   经纬度      56 non-null     object        
 3   jobid    56 non-null     object        
 4   薪资       56 non-null     object        
 5   城市       56 non-null     object        
 6   学历要求     56 non-null     object        
 7   经验要求     56 non-null     object        
 8   公司性质     56 non-null     object        
 9   行业       56 non-null     object        
 10  发布时间     56 non-null     datetime64[ns]
 11  薪资下限(k)  49 non-null     float64       
 12  薪资上限(k)  49 non-null     

{'json': 'C:\\Users\\Buer_vakabauta\\Desktop\\Course\\data_analysis\\src\\washdata\\cleaned_jobs_20251228_211500.json',
 'csv': 'C:\\Users\\Buer_vakabauta\\Desktop\\Course\\data_analysis\\src\\washdata\\cleaned_jobs_20251228_211500.csv',
 'excel': None}

In [51]:
clean_salary(test_data)

{'min_salary': 1.2, 'max_salary': 1.3, 'avg_salary': 1.3}